# Otsu Thresholding: Flood Mapping Baseline

Tile-based Otsu thresholding on VV polarization for flood extent mapping.

Applied to the same test tiles as U-Net and RF for direct comparison.

**Method:** For each tile, an optimal threshold is automatically determined
by maximizing inter-class variance (Otsu, 1979; Lee & Jurkevich, 1989).
Tiles with < 1% water in ground truth are excluded, consistent with
the STURM-Flood dataset filtering (Notarangelo et al., 2025).

**References:**
- Otsu, N. (1979). A threshold selection method from gray-level histograms.
  IEEE Trans. Syst. Man Cybern., 9, 62–66.
- Lee, J.S. & Jurkevich, I. (1989). Segmentation of SAR images.
  IEEE Trans. Geosci. Remote Sens., 27(6), 674–680.
- Notarangelo et al. (2025). STURM-Flood. Big Earth Data, 9(3), 412–438.

## Cell 1: Configuration
Mount Drive and configure paths and output folders.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# ============================================================
# OUTPUT FOLDER STRUCTURE
# Mirrors the RF results folder for easy cross-method comparison.
# ============================================================
output_base = '/content/drive/MyDrive/Examensarbete/Otsu_results_nofilter/goteborg_final'
dirs = {
    'tables':  os.path.join(output_base, 'tables'),
    'geotiff': os.path.join(output_base, 'geotiff_predictions'),
}
for d in dirs.values():
    os.makedirs(d, exist_ok=True)
print("Output folders ready:")
for k, v in dirs.items():
    print(f"  {k}: {v}")

# ============================================================
# PATHS TO TEST DATA
# Same test tiles used for U-Net and RF evaluation.
# ============================================================
# S1 tiles (normalized 0-1, VV band 1, VH band 2)
test_s1_dir  = '/content/drive/MyDrive/Examensarbete/Dataset_goteborg_final/Sentinel1/S1/'
# Ground truth flood masks
test_mask_dir = '/content/drive/MyDrive/Examensarbete/Dataset_goteborg_final/Sentinel1/Floodmaps/'

# ============================================================
# FILTER SETTINGS
# Tiles with < min_water_pct water pixels in ground truth
# are excluded. Set to 1% to match STURM-Flood dataset
# filtering (Notarangelo et al., 2025, p. 419).
# ============================================================
min_water_pct = 0.0  # percent (not fraction)

# Path to tile_index.csv from RF run (optional, for updating cross-method index)
#tile_index_path = '/content/drive/MyDrive/Examensarbete/RF_results/tables/tile_index.csv'


## Cell 2: Imports

In [ ]:
# !pip install rasterio scikit-image tqdm --quiet

import os
import glob
from datetime import datetime

import numpy as np
import pandas as pd
import rasterio
from skimage.filters import threshold_otsu
from sklearn.metrics import confusion_matrix
from tqdm import tqdm


## Cell 3: Load and verify test tiles

In [ ]:
# Scan test S1 tiles
test_s1_files = sorted(glob.glob(os.path.join(test_s1_dir, '*.tif')))
print(f"S1 tiles found: {len(test_s1_files)}")

# Build mask lookup: filename -> mask path
test_mask_files = sorted(glob.glob(os.path.join(test_mask_dir, '*.tif')))
mask_lookup = {os.path.basename(p): p for p in test_mask_files}
print(f"Mask files found: {len(test_mask_files)}")

# Verify matching
matched = [f for f in test_s1_files if os.path.basename(f) in mask_lookup]
print(f"Matched tile+mask pairs: {len(matched)}")

if len(matched) == 0:
    raise ValueError("No matched tile+mask pairs found. Check your paths.")


## Cell 4: Helper functions

In [ ]:
def compute_water_pct(mask_path):
    """
    Compute the percentage of water pixels in a ground truth mask.
    Used to filter out tiles with < min_water_pct water coverage.

    Args:
        mask_path: Path to ground truth GeoTIFF mask.

    Returns:
        Float: percentage of water pixels (0-100).
    """
    with rasterio.open(mask_path) as src:
        mask = src.read(1).flatten()
    valid = mask != 99  # exclude nodata
    if valid.sum() == 0:
        return 0.0
    return (mask[valid] > 0).sum() / valid.sum() * 100


def otsu_classify_tile(s1_path):
    """
    Apply Otsu thresholding to the VV band (band 1) of a tile.

    The VV band is used because it provides the best contrast between
    smooth water surfaces and rougher land in C-band SAR imagery
    (Notarangelo et al., 2025; Amitrano et al., 2024).

    Otsu's method automatically selects the threshold that maximizes
    inter-class variance, separating water (low backscatter) from
    land (high backscatter) pixels (Otsu, 1979).

    Tile-based (local) thresholding is used rather than global
    thresholding to account for spatial variation in backscatter
    across the scene (Lee & Jurkevich, 1989).

    Args:
        s1_path: Path to normalized S1 GeoTIFF tile (band 1 = VV).

    Returns:
        pred:   (128*128,) uint8 array, 0=land, 1=water, 255=nodata
        thresh: float, the Otsu threshold value (in normalized 0-1 range)
        profile: rasterio profile for GeoTIFF export
    """
    with rasterio.open(s1_path) as src:
        vv = src.read(1)  # VV polarization, band 1
        profile = src.profile.copy()

    vv_flat = vv.flatten()

    # Identify valid (finite) pixels
    valid = np.isfinite(vv_flat)

    # Initialize output with nodata
    pred = np.full(len(vv_flat), 255, dtype=np.uint8)

    if valid.sum() == 0:
        return pred, np.nan, profile

    vv_valid = vv_flat[valid]

    # Compute Otsu threshold on VV band
    # Low backscatter (low values) = water
    # High backscatter (high values) = land
    thresh = threshold_otsu(vv_valid)

    # Classify: pixels below threshold = water (1), above = land (0)
    pred[valid] = (vv_valid < thresh).astype(np.uint8)

    return pred, float(thresh), profile


def evaluate_tile(pred_flat, mask_path):
    """
    Pixel-level evaluation of Otsu prediction against ground truth.

    Args:
        pred_flat: (N,) uint8 array of predictions (0=land, 1=water, 255=nodata)
        mask_path: Path to ground truth mask GeoTIFF.

    Returns:
        Dict with tp, tn, fp, fn, or None if no valid pixels.
    """
    with rasterio.open(mask_path) as src:
        mask = src.read(1).flatten()

    yt = (mask > 0).astype(np.uint8)
    yp = pred_flat

    # Only evaluate on valid pixels (not nodata in either)
    valid = (mask != 99) & (yp != 255)
    if valid.sum() == 0:
        return None

    yt = yt[valid]
    yp = yp[valid]

    cm = confusion_matrix(yt, yp, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    return {'tp': int(tp), 'tn': int(tn), 'fp': int(fp), 'fn': int(fn)}


def append_result_to_csv(csv_path, row_dict):
    """Append one result row to CSV. Creates file with headers if needed."""
    df_row = pd.DataFrame([row_dict])
    if os.path.exists(csv_path):
        df_row.to_csv(csv_path, mode='a', header=False, index=False)
    else:
        df_row.to_csv(csv_path, mode='w', header=True, index=False)


## Cell 5: Run Otsu on all test tiles

For each tile:
1. Check ground truth water percentage, skip if < 1% (Notarangelo et al., 2025)
2. Apply Otsu threshold on VV band
3. Evaluate against ground truth
4. Log results to CSV
5. Export GeoTIFF prediction for QGIS

In [ ]:
results_csv  = os.path.join(dirs['tables'], 'otsu_results.csv')
tile_index_rows = []

# Accumulators for aggregate metrics
all_tp, all_tn, all_fp, all_fn = 0, 0, 0, 0
n_processed  = 0
n_skipped    = 0

print("=" * 60)
print("  OTSU THRESHOLDING | VV BAND, PER TILE")
print(f"  Test tiles: {len(matched)}")
print(f"  Min water filter: {min_water_pct}%")
print("=" * 60)

for s1_path in tqdm(matched, desc="Processing tiles"):
    fname    = os.path.basename(s1_path)
    mask_path = mask_lookup[fname]

    # --- Filter: skip tiles with < min_water_pct water in ground truth ---
    # Consistent with STURM-Flood dataset filtering (Notarangelo et al., 2025, p. 419).
    # Tiles without water cannot contribute to meaningful flood detection
    # and would introduce arbitrary thresholds on unimodal histograms.
    water_pct = compute_water_pct(mask_path)
    if water_pct < min_water_pct:
        n_skipped += 1
        continue

    # --- Apply Otsu thresholding ---
    pred, thresh, profile = otsu_classify_tile(s1_path)

    # --- Evaluate against ground truth ---
    metrics = evaluate_tile(pred, mask_path)
    if metrics is None:
        n_skipped += 1
        continue

    tp, tn, fp, fn = metrics['tp'], metrics['tn'], metrics['fp'], metrics['fn']
    all_tp += tp; all_tn += tn; all_fp += fp; all_fn += fn
    n_processed += 1

    # Per-tile metrics
    pw  = tp/(tp+fp)     if (tp+fp)   else 0.0
    rw  = tp/(tp+fn)     if (tp+fn)   else 0.0
    fw  = 2*pw*rw/(pw+rw) if (pw+rw) else 0.0
    pn  = tn/(tn+fn)     if (tn+fn)   else 0.0
    rn  = tn/(tn+fp)     if (tn+fp)   else 0.0
    fn_ = 2*pn*rn/(pn+rn) if (pn+rn) else 0.0
    acc = (tp+tn)/(tp+tn+fp+fn)

    # --- Save GeoTIFF prediction ---
    out_path = os.path.join(dirs['geotiff'], f"Otsu_{fname}")
    profile.update(count=1, dtype='uint8', nodata=255)
    with rasterio.open(out_path, 'w', **profile) as dst:
        dst.write(pred.reshape(128, 128), 1)

    # --- Log to CSV ---
    row = {
        'timestamp':       datetime.now().isoformat(),
        'tile_id':         fname,
        'water_pct_gt':    round(water_pct, 2),
        'otsu_threshold':  round(thresh, 4),
        'min_water_filter': min_water_pct,
        'channel':         'VV',
        'accuracy':        round(acc, 6),
        'water_precision': round(pw, 6),
        'water_recall':    round(rw, 6),
        'water_f1':        round(fw, 6),
        'nw_precision':    round(pn, 6),
        'nw_recall':       round(rn, 6),
        'nw_f1':           round(fn_, 6),
        'macro_f1':        round((fw + fn_) / 2, 6),
        'tp': tp, 'tn': tn, 'fp': fp, 'fn': fn,
        'otsu_prediction': out_path,
    }
    append_result_to_csv(results_csv, row)

    # For tile index
    tile_index_rows.append({
        'tile_id':         fname,
        'source_tile':     s1_path,
        'mask':            mask_path,
        'otsu_prediction': out_path,
    })

print(f"\nDone. Processed: {n_processed} tiles, Skipped: {n_skipped} tiles")
print(f"Results saved to {results_csv}")


## Cell 6: Aggregate results

In [ ]:
# Compute aggregate metrics across all processed tiles
# (same approach as STURM-Flood Table 11)
pw  = all_tp/(all_tp+all_fp)     if (all_tp+all_fp)   else 0.0
rw  = all_tp/(all_tp+all_fn)     if (all_tp+all_fn)   else 0.0
fw  = 2*pw*rw/(pw+rw)            if (pw+rw)           else 0.0
pn  = all_tn/(all_tn+all_fn)     if (all_tn+all_fn)   else 0.0
rn  = all_tn/(all_tn+all_fp)     if (all_tn+all_fp)   else 0.0
fn_ = 2*pn*rn/(pn+rn)            if (pn+rn)           else 0.0
acc = (all_tp+all_tn)/(all_tp+all_tn+all_fp+all_fn)

print("=" * 60)
print("  AGGREGATE RESULTS | OTSU THRESHOLDING")
print(f"  Tiles processed: {n_processed}  |  Skipped: {n_skipped}")
print("=" * 60)
print(f"  Accuracy:         {acc:.4f}")
print(f"  Macro F1:         {(fw+fn_)/2:.4f}")
print(f"  Water Precision:  {pw:.4f}")
print(f"  Water Recall:     {rw:.4f}")
print(f"  Water F1:         {fw:.4f}")
print(f"  TP={all_tp:,}  TN={all_tn:,}  FP={all_fp:,}  FN={all_fn:,}")
print("=" * 60)

# Compare with STURM-Flood Otsu baseline (Notarangelo et al., 2025, Table 11)
print("\n  STURM-Flood Otsu baseline (Notarangelo et al., 2025):")
print(f"  Water Precision: 0.2522  |  Water Recall: 0.7776  |  Water F1: 0.3808")

# Save aggregate summary
summary_path = os.path.join(dirs['tables'], 'otsu_summary.csv')
summary = {
    'timestamp':        datetime.now().isoformat(),
    'n_tiles':          n_processed,
    'n_skipped':        n_skipped,
    'min_water_filter': min_water_pct,
    'channel':          'VV',
    'accuracy':         round(acc, 6),
    'macro_f1':         round((fw+fn_)/2, 6),
    'water_precision':  round(pw, 6),
    'water_recall':     round(rw, 6),
    'water_f1':         round(fw, 6),
    'nw_precision':     round(pn, 6),
    'nw_recall':        round(rn, 6),
    'nw_f1':            round(fn_, 6),
    'tp': all_tp, 'tn': all_tn, 'fp': all_fp, 'fn': all_fn,
}
pd.DataFrame([summary]).to_csv(summary_path, index=False)
print(f"\nSummary saved to {summary_path}")

# Combine summary row + per-tile results into one CSV (same format as U-Net)
per_tile_df = pd.read_csv(results_csv)
summary_df = pd.DataFrame([summary])
combined_df = pd.concat([summary_df, per_tile_df], ignore_index=True)
combined_path = os.path.join(dirs['tables'], 'otsu_results_combined.csv')
combined_df.to_csv(combined_path, index=False)
print(f"Combined saved to {combined_path}")
